# Cross integration: Multiple RNA + ATAC

Cross integration combines batches that all measure the same modalities. The task is to remove batch effects and keep the biological structure. This tutorial is for batches that each hold RNA and ATAC peaks. It runs StabMap and scMM on `D56mini`, 4,500 cells in three batches, cut from the first three batches of the benchmark dataset `D56`, and then on data in your own format.

Methods run on Linux. On macOS or Windows, [open this notebook in Colab](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_cross_rna_atac.ipynb): sections 1 to 4 take about 8 minutes there.

## 1. Install

pip may then print a note about restarting the kernel: no restart is needed, because the pins keep the numpy and pandas already installed.

In [ ]:
import numpy, pandas
pins = f"numpy=={numpy.__version__} pandas=={pandas.__version__}"
%pip install -q "multibench-sc>=0.6.0" {pins}

## 2. Download the data and the environments

`mtb.data.fetch` downloads `D56mini` (10 MB) once. `mtb.env.install` downloads the environments for StabMap and scMM. The download is 1.4 GB.

In [ ]:
import multibench as mtb
import pandas as pd

pd.set_option("display.max_columns", None)   # show every column of the tables

mtb.data.fetch("D56mini")
METHODS = ["StabMap", "scMM"]
envs = mtb.env.install(METHODS)

## 3. Run the methods

`run` runs each method on `D56mini` and [scores its output](https://dsichang.github.io/scMultiBench/tutorials/evaluate/#what-each-metric-measures). The clustering metrics score how well the embedding's clusters and neighbourhoods match the cell types, and how separate the clusters are. The batch metrics measure how well the batches mix. The classification scores measure how well the cell types learned on one batch are predicted in the other batches. Higher is better for every metric.

<details>
<summary>Details</summary>

`params={"Method": {"key": value}}` sets a method's parameters, and `mtb.params_for` lists them.

`mtb.load_run("out/D56mini")` reloads these results later.

</details>

In [ ]:
res = mtb.run("D56mini", "cross", methods=METHODS, out_dir="out/D56mini")
res.summary

## 4. Plot

`res.plot()` draws the scores as a bubble table. Circle size ranks the rows within each column and colour shows the value scaled per column, so a small gap can look large. `res.summary` has the values.

<details>
<summary>Details</summary>

Metrics are grouped by family: blue for clustering, green for batch correction, pink for classification. Each family starts with an `Overall` bar: each row's mean rank over the family's metrics, scaled so that the lowest gets the shortest bar.

A column with fewer than two different values is drawn grey. A figure of one method is grey everywhere.

</details>

In [ ]:
res.plot()

## 5. Your own data

Your data needs raw RNA counts in one `.h5ad` file and an ATAC peak matrix of the same cells in another, with a cell type and a batch for each cell. Give the same peaks in every batch.

The methods of this task read three batches, and UINMF also reads two.

Recipes cover [a 10x `.h5` file](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-10x-h5), [labels in a CSV file](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-labels-csv), [no labels](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-no-labels) and [another number of donors](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-donors).

<details>
<summary>Details: export</summary>

The names in quotes say where your file holds each part. `"X"` is the main matrix, `"obsm:atac"` the matrix `obsm["atac"]`, `"layer:counts"` the layer `layers["counts"]`, and `"obs:celltype"` the column `obs["celltype"]`. A wrong name after `obsm:`, `layer:` or `obs:` raises an error that lists the names the file has.

The batches are numbered in the sorted order of their names, read as text: donors `A`, `B` and `C` become batches 1, 2 and 3, and `10` comes before `2`.

For one AnnData per batch, call `export_dataset` once per batch with `batch_index=N` instead of `batch=`.

</details>

The next cell writes two small example files, `my_rna.h5ad` and `my_atac.h5ad`, from 60% of the cells of `D56mini`, with a batch column. Skip it for your own data.

In [ ]:
mtb.data.write_example("D56mini")

Read your files. For your own data, change every file name marked `# <-`. On Colab, upload the files first: click the folder icon on the left, then the upload icon.

In [ ]:
import anndata as ad

rna = ad.read_h5ad("my_rna.h5ad")     # <- your RNA file, every batch in it
atac = ad.read_h5ad("my_atac.h5ad")   # <- your peak file, the same cells
print(rna)
print(atac)

Write the dataset folder. Change each name in quotes marked `# <-` to one the printout above lists for your files, with its prefix, as `"obs:celltype"`.

In [ ]:
NAME = "MYCROSS_RNA_ATAC"   # <- a new name for your own data
mtb.io.export_dataset(rna, "mydata/" + NAME,
                      rna="X",                # <- where the RNA counts are:
                                              #    "X" or "layer:counts"
                      atac=atac,              # the ATAC file, peak counts in X
                      atac_kind="peak",       # the methods here read peaks
                      labels="obs:celltype",  # <- the cell-type column
                      batch="obs:batch",      # <- the batch or donor column
                      category="cross", overwrite=True)
mtb.scan("mydata/" + NAME, "cross", methods=METHODS)

In [ ]:
mine = mtb.run("mydata/" + NAME, "cross", methods=METHODS,
               out_dir="out/" + NAME)
mine.summary

In [ ]:
mine.plot()

## 6. Every method of this task

8 methods run this task. [This notebook](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_cross_rna_atac_all.ipynb) runs them all. Its 6 environments download 19.6 GB on a computer without a GPU and 23.0 GB on one with a GPU.

## Troubleshooting

`res.summary[res.summary.status != "scored"]` lists each method that failed or was skipped. Its `reason` column ends with the method's error output.

<details>
<summary>Details</summary>

| symptom | fix |
|---|---|
| a method fails on a small dataset | `mtb.scan` names what the method needs from the data's size in its `caveat` column |
| batch metrics use the wrong batches | `res.rescore(batch=...)`, with one batch per cell, scores again without running the methods |

</details>

## Next steps

- [Run a method](https://dsichang.github.io/scMultiBench/tutorials/run/), [Score an embedding](https://dsichang.github.io/scMultiBench/tutorials/evaluate/), [Load and plot results](https://dsichang.github.io/scMultiBench/tutorials/plot/) and [Discover methods](https://dsichang.github.io/scMultiBench/tutorials/discover/).
- The [interactive explorer](https://shiny.maths.usyd.edu.au/scMultiBench/) has the full benchmark's rankings, with no install needed.